# MRO Parts — Attribute Extraction Pipeline

**Input:** `data/cleaned_data.csv` (292,640 rows × 22 columns)  
**Output:** `data/processed/parts_with_attrs.csv`

## Philosophy

This dataset spans 20+ attribute families. Parts in one family look nothing like parts in another. **Every extractor is scoped to a specific `attribute_family` — no pattern is assumed to generalize across families without verification.**

Each family section follows the same three steps:
1. **Diagnose** — sample the descriptions, understand what's actually there
2. **Extract** — write patterns for what we observed, nothing more
3. **Audit** — report coverage and spot-check results before accepting them

A null extraction is better than a wrong one.

In [ ]:
import pandas as pd
import numpy as np
import re
import random
from collections import Counter

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 40)

CLEAN_PATH = 'data/cleaned_data.csv'
OUT_PATH   = 'data/processed/parts_with_attrs.csv'

print('Imports OK')

In [ ]:
# ---
# Load and strip to only the columns we will use for extraction.
# mro_description and livhaven_description are marketing boilerplate (distributor SEO copy).
# livhaven_short_description is a near-duplicate of default_short_description.
# Neither adds extraction value — confirmed by manual inspection.
# ---

raw = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False)
raw = raw.replace('', pd.NA)  # normalize empty strings

CORE_COLS = [
    'sku',
    'brand_name',
    'manufacturer_part_number',
    'attribute_family',
    'category_name',
    'default_name',
    'default_short_description',   # primary spec source — comma-separated attributes for most families
    'manufacturer_description',    # authentic technical prose — only 13% filled but high quality
    'last_sold_price',
    'item_weight',
]

df = raw[CORE_COLS].copy()

print(f'Loaded: {len(df):,} rows')
print(f'\nFill rates on extraction sources:')
for col in ['default_short_description', 'manufacturer_description']:
    pct = df[col].notna().mean() * 100
    print(f'  {col:<35} {pct:.1f}%')

print(f'\nAttribute families:')
print(df['attribute_family'].value_counts().head(20).to_string())

---
## Extraction Framework

A `FamilyExtractor` holds all the regex patterns for one attribute family.
It only runs on rows where `attribute_family` matches — it never touches other families.

Each pattern:
- Is named explicitly (what attribute it extracts)
- Has a `source_col` (which description column it searches)
- Returns `pd.NA` on no match — not a guess, not a default

After extraction, `coverage_report()` shows exactly how many rows each pattern fired on.

In [ ]:
class FamilyExtractor:
    """
    Scoped attribute extractor for a single attribute_family.
    Patterns are defined explicitly per family — no cross-family sharing.
    """

    def __init__(self, family_name: str):
        self.family = family_name
        # List of (attr_name, compiled_regex, source_col, transform_fn)
        self._patterns: list[tuple] = []

    def add(self, attr_name: str, pattern: str, source_col: str = 'default_short_description',
            flags=re.IGNORECASE, group: int = 0, transform=None):
        """
        attr_name:  column name in the output dataframe
        pattern:    regex string — use capturing groups if you want a substring, group=0 for full match
        source_col: which description column to search
        group:      which regex group to return (0 = full match)
        transform:  optional callable to clean the matched string
        """
        self._patterns.append(
            (attr_name, re.compile(pattern, flags), source_col, group, transform)
        )

    def _extract_row(self, row: pd.Series) -> dict:
        result = {}
        for attr, pat, src_col, grp, transform in self._patterns:
            text = row.get(src_col, pd.NA)
            if pd.isna(text):
                result[attr] = pd.NA
                continue
            m = pat.search(str(text))
            if m:
                val = m.group(grp) if grp else m.group(0)
                val = val.strip()
                if transform:
                    val = transform(val)
                result[attr] = val
            else:
                result[attr] = pd.NA
        return result

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Run extraction on the subset of df that matches this family. Returns that subset with new columns."""
        subset = df[df['attribute_family'] == self.family].copy()
        attr_names = [p[0] for p in self._patterns]
        extracted = subset.apply(self._extract_row, axis=1, result_type='expand')
        for col in attr_names:
            subset[col] = extracted[col]
        return subset

    def coverage_report(self, result_df: pd.DataFrame) -> pd.DataFrame:
        """Show fill rate for every extracted attribute."""
        attr_names = [p[0] for p in self._patterns]
        rows = []
        total = len(result_df)
        for attr in attr_names:
            if attr not in result_df.columns:
                continue
            n_filled = result_df[attr].notna().sum()
            rows.append({
                'attribute': attr,
                'extracted': n_filled,
                'pct_of_family': f'{n_filled/total*100:.1f}%',
                'sample': result_df[attr].dropna().iloc[0] if n_filled > 0 else '—',
            })
        return pd.DataFrame(rows).set_index('attribute')


def spot_check(result_df: pd.DataFrame, attr: str, n: int = 10, seed: int = 42):
    """Print source text alongside extracted value — use this to verify patterns are correct."""
    hits = result_df[result_df[attr].notna()][['default_short_description', attr]]
    sample = hits.sample(min(n, len(hits)), random_state=seed)
    for _, row in sample.iterrows():
        print(f'  SRC: {str(row["default_short_description"])[:120]}')
        print(f'  → {attr}: {row[attr]}')
        print()


print('Framework loaded.')

---
## Family 1: Valves and Accessories
**78,355 rows — largest family in the dataset**

### Diagnostic

Not all valve rows have parseable spec text. Three distinct description types exist:

| Type | Description | Approx % of family |
|---|---|---|
| **Structured spec list** | `"3-Way, NBR Seals, 24VDC, 0-125PSI"` | ~35% |
| **Model code / part reference** | `"Similar To: 4WE10Y3X/..."`, `"Aluminum Valve Manifold AD03..."` | ~42% |
| **Plain name only** | `"Pneumatic Directional Valve"` | ~48% (overlaps above) |

**Extraction is only reliable on the structured spec list rows.** We mark non-parseable rows but do not attempt to force extraction — that would produce garbage.

In [ ]:
# --- VALVE DIAGNOSTIC ---
# Understand the distribution of description types before writing any extractors.

valve_df = df[df['attribute_family'] == 'Valves_and_Accessories'].copy()
print(f'Valve rows: {len(valve_df):,}')
print(f'Have short_desc: {valve_df["default_short_description"].notna().sum():,}')
print()

desc = valve_df['default_short_description'].dropna()

# Identify rows with actual structured specs (multiple comma-separated attributes + known keywords)
SPEC_KEYWORDS = r'(Way|Position|Solenoid|NPT|BSPT|BSP|Seals|Body|VDC|VAC|PSI|Cv\b)'
has_specs = desc.str.contains(SPEC_KEYWORDS, regex=True, case=False, na=False)

# Identify rows that are model-code/reference noise
is_reference = desc.str.match(
    r'^(Part\s*#|Similar To|Also Known As|Model Code|Aluminum Valve|Ductile Valve|HANDLE|Locking Device)',
    case=False, na=False
)

print(f'Structured spec rows (parseable):  {has_specs.sum():,} ({has_specs.sum()/len(valve_df)*100:.1f}% of family)')
print(f'Model-code / reference rows:       {is_reference.sum():,} ({is_reference.sum()/len(valve_df)*100:.1f}% of family)')
print()

# Sample 8 structured ones so we can verify our patterns are targeting the right text
print('=== Sample structured spec rows ===')
for d in desc[has_specs & ~is_reference].sample(8, random_state=42):
    print(f'  {d[:160]}')

In [ ]:
# --- VALVE EXTRACTORS ---
# Patterns derived from the diagnostic above.
# Each pattern was verified against actual samples before being added.

valve_ex = FamilyExtractor('Valves_and_Accessories')

# Flag: does this row have structured spec text at all?
# Rows without this flag should be treated as low-information for attribute matching.
valve_ex.add(
    'has_structured_specs',
    r'(\d+-Way|\d+-Position|Solenoid|NPT|BSPT|\d+VDC|\d+VAC|PSI\b|\bCv\b)',
)

# Number of flow paths — "3-Way", "4-Way" etc.
valve_ex.add(
    'ways',
    r'(\d+)-Way',
    group=1,
)

# Number of positions — "2-Position"
valve_ex.add(
    'positions',
    r'(\d+)-Position',
    group=1,
)

# Solenoid configuration
valve_ex.add(
    'solenoid_type',
    r'(Single|Double|External|Internal)\s*Solenoid',
    group=1,
    transform=str.title,
)

# Return mechanism
valve_ex.add(
    'return_type',
    r'(Spring Return|Detent|Air Return|Spring Offset)',
    transform=str.title,
)

# Port thread type — NPT, BSPT, BSP
# Captures the size+standard together, e.g. "1/4\" NPT", "R3/8\" BSPT", "M5 X 0.8"
valve_ex.add(
    'port_thread',
    r'(?:R|G)?\d+/\d+["\']?\s*(?:NPT|BSPT|BSP)|M\d+\s*[Xx]\s*[\d.]+',
)

# Seal material — only extract the material name, drop the word 'Seals'
valve_ex.add(
    'seal_material',
    r'(NBR|FKM|PTFE|Polyurethane|Buna-N|EPDM|Viton)\s*Seals',
    group=1,
    transform=str.upper,
)

# Body material — the word before 'Body'
# CAUTION: "Aluminum Valve Body" vs "Thermoset Epoxy Body" — capture full phrase before Body
valve_ex.add(
    'body_material',
    r'(Brass|Aluminum(?:\s*Diecast)?|Zinc|Stainless\s*Steel|Thermoset\s*Epoxy|Ductile\s*Iron|Nylon|Polypropylene)\s*Body',
    group=1,
    transform=str.title,
)

# Voltage — e.g. "24VDC", "120VAC", "24VAC/DC"
# Note: some rows have "24Vdc" (mixed case) — IGNORECASE handles this
valve_ex.add(
    'voltage',
    r'(\d+(?:\.\d+)?)\s*(VDC|VAC(?:/DC)?|Vdc|Vac)',
    transform=lambda x: x.upper().replace(' ', ''),
)

# Pressure range — extract the max value (most useful for comparison)
# Pattern: "3-150PSI" → capture "150", "28\"Hg-30PSI" → capture "30"
valve_ex.add(
    'pressure_max_psi',
    r'[\d.]+["\']?Hg-([\d.]+)\s*PSI|(?:[\d.]+)-([\d.]+)\s*PSI',
    group=0,  # keep full match — parse further if needed
    transform=lambda x: x.strip(),
)

# Cv flow coefficient — e.g. "2.00 Cv"
valve_ex.add(
    'cv_flow',
    r'([\d.]+)\s*Cv\b',
    group=1,
)

# Mounting type — manifold, inline, subbase
valve_ex.add(
    'mounting_type',
    r'(Manifold|Inline|Sub-?base|Base|Surface)\s*(?:Mount(?:ing)?|Type)?',
    transform=str.title,
)

print('Valve extractors defined.')
print(f'Patterns: {len(valve_ex._patterns)}')

In [ ]:
# --- RUN VALVE EXTRACTION ---

valve_result = valve_ex.apply(df)

print('=== Valve Coverage Report ===')
print(valve_ex.coverage_report(valve_result).to_string())
print()

In [ ]:
# --- VALVE SPOT CHECKS ---
# Verify each extractor is pulling the right thing before trusting coverage numbers.
# If any sample looks wrong, go back and fix the pattern.

for attr in ['ways', 'voltage', 'seal_material', 'body_material', 'pressure_max_psi', 'port_thread']:
    print(f'=== {attr} ===')
    spot_check(valve_result, attr, n=5)
    print()

---
## Family 2: Cylinders and Accessories
**63,929 rows — second largest family**

### Diagnostic

Cylinders are the most consistently structured family in the dataset. ~93% of rows have parseable short descriptions in a predictable natural-language format:

```
Double-acting non-lube NFPA tie-rod pneumatic cylinder with piston magnet +
"LipSeal" piston + small male rod-end thread - 2" bore (B2") - 5" stroke (S5") -
2 x 3/8" NPT threaded pneumatic ports - 3/4"-16 UNF male threaded rod-end
```

Key attributes explicitly labeled:
- Bore: `N" bore` or `(BN")`
- Stroke: `N" stroke` or `(SN")`
- Port size: `N/N" NPT`
- Acting type: `Double-acting` / `Single-acting`
- Construction: `NFPA tie-rod`, `compact`, `stainless steel`
- Piston magnet: present/absent explicitly stated

In [ ]:
# --- CYLINDER DIAGNOSTIC ---

cyl_df = df[df['attribute_family'] == 'Cylinders_and_Accessories'].copy()
print(f'Cylinder rows: {len(cyl_df):,}')
print(f'Have short_desc: {cyl_df["default_short_description"].notna().sum():,}')
print()

cyl_desc = cyl_df['default_short_description'].dropna()

checks = {
    'has bore spec':   r"\b\d[\d/\-]*['\"\s]*bore",
    'has stroke spec': r"\b\d[\d/\-]*['\"\s]*stroke",
    'has NPT port':    r'\bNPT\b',
    'has acting type': r'(double|single)[- ]acting',
    'has magnet flag': r'piston magnet',
}

for label, pat in checks.items():
    n = cyl_desc.str.contains(pat, case=False, regex=True, na=False).sum()
    print(f'  {label:<25} {n:,} ({n/len(cyl_df)*100:.1f}%)')

print()
print('=== Sample rows ===')
for d in cyl_desc.sample(6, random_state=42):
    print(f'  {d[:180]}')
    print()

In [ ]:
# --- CYLINDER EXTRACTORS ---

cyl_ex = FamilyExtractor('Cylinders_and_Accessories')

# Acting type — almost always at start of description
cyl_ex.add(
    'acting_type',
    r'(Double|Single)[- ]acting',
    group=1,
    transform=str.title,
)

# Construction type — these are distinct and non-overlapping in practice
cyl_ex.add(
    'construction_type',
    r'(NFPA\s*tie-rod|stainless\s*steel|compact|universal\s*midget|roundline|mill-type|welded)',
    transform=str.title,
)

# Bore diameter — format: "2\" bore" or "(B2\")" or "B1-1/2\""
# Captures the numeric/fractional value, normalizes to string
cyl_ex.add(
    'bore_inch',
    r'([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)[\'\"\s]+bore|\(B([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)["\']?\)',
    group=0,  # full match — bore value can be parsed from group 1 or 2
    transform=lambda x: re.search(r'([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)', x).group(1) if re.search(r'([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)', x) else pd.NA,
)

# Stroke length — format: "5\" stroke" or "(S5\")"  
cyl_ex.add(
    'stroke_inch',
    r'([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)[\'\"\s]+stroke|\(S([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)["\']?\)',
    group=0,
    transform=lambda x: re.search(r'([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)', x).group(1) if re.search(r'([\d]+(?:-[\d]+/[\d]+|/[\d]+)?)', x) else pd.NA,
)

# Port size — e.g. "1/4\" NPT", "3/8\" NPT"
cyl_ex.add(
    'port_size_npt',
    r'(\d+/\d+)["\']\s*NPT',
    group=1,
)

# Rod end thread — e.g. "5/8\"-11 UNC", "3/4\"-16 UNF", "M16 x 2"
cyl_ex.add(
    'rod_thread',
    r'(\d+/\d+["\']?-\d+\s*UN[CF]|M\d+\s*[Xx]\s*[\d.]+)\s*(?:male|female)?\s*threaded\s*rod',
    group=1,
)

# Piston magnet — boolean-style: presence = True
cyl_ex.add(
    'has_piston_magnet',
    r'piston\s*magnet',
    transform=lambda x: 'Yes',
)

# Cushion — location
cyl_ex.add(
    'cushion',
    r'cushion(?:ed)?\s*(?:at\s*)?(both\s*ends|head\s*end(?:\s*only)?|rod\s*end(?:\s*only)?)',
    group=1,
    transform=str.title,
)

# Seal / piston type
cyl_ex.add(
    'piston_seal',
    r'"?(LipSeal|O-Ring|Quad\s*Seal|Buna-N|PTFE)"?\s*piston',
    group=1,
    transform=str.title,
)

print('Cylinder extractors defined.')
print(f'Patterns: {len(cyl_ex._patterns)}')

In [ ]:
# --- RUN CYLINDER EXTRACTION ---

cyl_result = cyl_ex.apply(df)

print('=== Cylinder Coverage Report ===')
print(cyl_ex.coverage_report(cyl_result).to_string())
print()

In [ ]:
# --- CYLINDER SPOT CHECKS ---

for attr in ['bore_inch', 'stroke_inch', 'port_size_npt', 'acting_type', 'has_piston_magnet']:
    print(f'=== {attr} ===')
    spot_check(cyl_result, attr, n=5)
    print()

---
## Remaining Families — Scaffold

The families below have not yet been analyzed. Each needs its own diagnostic pass before any extractors are written.
Do not copy-paste patterns from Valves or Cylinders — the formats are different.

**Priority order by row count:**
1. `Hydraulic_Filters_and_Fluid_Conditioning` — 31,692 rows
2. `Air_Preparation` — 12,696 rows
3. `Sensors_and_Accessories` — 12,540 rows  ← highly variable format, diagnose carefully
4. `Electrical_Panel_Components` — 11,805 rows
5. `Fittings` — 13,213 rows
6. `Lubrication_Systems_and_Accessories` — 8,924 rows
7. All others

In [ ]:
# SCAFFOLD — Hydraulic Filters
# Run this diagnostic first, then write extractors

fam = 'Hydraulic_Filters_and_Fluid_Conditioning'
sub = df[df['attribute_family'] == fam]
print(f'{fam}: {len(sub):,} rows')
print(f'Have short_desc: {sub["default_short_description"].notna().sum():,}')
print()
print('=== 10 random samples ===')
for d in sub['default_short_description'].dropna().sample(10, random_state=42):
    print(f'  {d[:180]}')
    print()

# TODO: after reviewing samples above, add a HydraulicFilters FamilyExtractor here

In [ ]:
# SCAFFOLD — Sensors and Accessories
# NOTE: from initial inspection, this family has highly variable formats.
# Some rows have structured specs, many are model-code only.
# Proceed carefully — diagnose before extracting.

fam = 'Sensors_and_Accessories'
sub = df[df['attribute_family'] == fam]
print(f'{fam}: {len(sub):,} rows')
print(f'Have short_desc: {sub["default_short_description"].notna().sum():,}')
print()
print('=== 10 random samples ===')
for d in sub['default_short_description'].dropna().sample(10, random_state=42):
    print(f'  {d[:180]}')
    print()

# TODO: after reviewing samples above, add a Sensors FamilyExtractor here

---
## Final Assembly — Combine and Save

Concatenate all family-specific result DataFrames.
Families that have not yet been processed are included as-is (no extracted attribute columns).
This lets the file grow incrementally as more families are done.

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

# Families with extractors complete
processed_families = {
    'Valves_and_Accessories':           valve_result,
    'Cylinders_and_Accessories':        cyl_result,
}

processed_skus = set()
for name, res in processed_families.items():
    processed_skus.update(res['sku'].tolist())

# Remaining families — include with no extracted columns yet
remainder = df[~df['sku'].isin(processed_skus)].copy()

# Concatenate — pandas will fill missing attribute columns with NaN for remainder rows
combined = pd.concat(
    list(processed_families.values()) + [remainder],
    ignore_index=True,
    sort=False,
)

combined.to_csv(OUT_PATH, index=False)

print(f'Saved → {OUT_PATH}')
print(f'Rows:    {len(combined):,}')
print(f'Columns: {len(combined.columns)}')
print()
print('Extracted attribute columns:')
base_cols = set(CORE_COLS)
attr_cols = [c for c in combined.columns if c not in base_cols]
for c in attr_cols:
    n = combined[c].notna().sum()
    print(f'  {c:<30} {n:,} filled ({n/len(combined)*100:.1f}%)')